# Mangrove encroachment risk

This notebook calculates the land-cover composition of the 100 m ring surrounding the Forces of Nature mangrove polygons. It then groups the detailed 2013 land-cover classes into the four encroachment classes used in the broader encroachment analysis.

In [ ]:
from pathlib import Path
import re

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display


In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
jamaica_metric_grid_crs = "EPSG:3448"

mangroves_fon_path = base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp"
land_use_path = base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp"

output_dir = base_path / "dphil_paper_3/processed_data/threats/encroachment_risk"
output_dir.mkdir(parents=True, exist_ok=True)

print(mangroves_fon_path)
print(land_use_path)
print(output_dir)


## Load source data

The Forces of Nature mangrove file is already in the Jamaica metric grid CRS. The 2013 land-cover layer is reprojected defensively so the area calculations are in square metres.

In [ ]:
mangroves_fon = gpd.read_file(mangroves_fon_path).to_crs(jamaica_metric_grid_crs)
mangroves_fon = mangroves_fon[mangroves_fon.geometry.notna() & ~mangroves_fon.geometry.is_empty].copy()
mangroves_fon["geometry"] = mangroves_fon.geometry.make_valid()

terrestrial_landcover = gpd.read_file(land_use_path).to_crs(jamaica_metric_grid_crs)
terrestrial_landcover = terrestrial_landcover[["Classify", "geometry"]].dropna(subset=["Classify"]).copy()
terrestrial_landcover = terrestrial_landcover[
    terrestrial_landcover.geometry.notna() & ~terrestrial_landcover.geometry.is_empty
].copy()

print(f"Mangrove features: {len(mangroves_fon):,}")
print(f"Land-cover features: {len(terrestrial_landcover):,}")
print(f"CRS: {terrestrial_landcover.crs}")


## Calculate 100 m surrounding ring

The ring is `mangrove_union.buffer(100).difference(mangrove_union)`, so the original mangrove footprint is excluded. Percentages labelled `% of surrounding ring` use the same denominator as `Encroachment.ipynb`: the land-cover-classified area within the ring.

In [ ]:
def summarise_surrounding_landuse(mangroves, landcover, distance_m):
    mangrove_union = mangroves.geometry.union_all()
    ring_geom = mangrove_union.buffer(distance_m).difference(mangrove_union)

    candidate_idx = list(landcover.sindex.query(ring_geom, predicate="intersects"))
    candidates = landcover.iloc[candidate_idx][["Classify", "geometry"]].copy()

    invalid_candidates = ~candidates.is_valid
    if invalid_candidates.any():
        candidates.loc[invalid_candidates, "geometry"] = candidates.loc[
            invalid_candidates, "geometry"
        ].make_valid()

    intersections = candidates.geometry.intersection(ring_geom)
    keep = intersections.notna() & ~intersections.is_empty

    land_in_ring = candidates.loc[keep, ["Classify"]].copy()
    land_in_ring["area_m2"] = intersections.loc[keep].area

    summary = (
        land_in_ring.groupby("Classify", as_index=False)["area_m2"]
        .sum()
        .sort_values("area_m2", ascending=False)
    )

    total_classified_area_m2 = summary["area_m2"].sum()
    ring_area_m2 = ring_geom.area

    summary["area_km2"] = summary["area_m2"] / 1e6
    summary["% of surrounding ring"] = np.where(
        total_classified_area_m2 > 0,
        summary["area_m2"] / total_classified_area_m2 * 100,
        0,
    )
    summary["% of geometric ring"] = np.where(
        ring_area_m2 > 0,
        summary["area_m2"] / ring_area_m2 * 100,
        0,
    )
    summary["distance_m"] = distance_m

    return summary, ring_area_m2, total_classified_area_m2


In [ ]:
summaries = {}
ring_area_by_distance = {}
classified_area_by_distance = {}

for distance in [100]:
    summary, ring_area_m2, classified_area_m2 = summarise_surrounding_landuse(
        mangroves_fon,
        terrestrial_landcover,
        distance,
    )
    summaries[distance] = summary
    ring_area_by_distance[distance] = ring_area_m2
    classified_area_by_distance[distance] = classified_area_m2

    print(f"{distance} m ring area: {ring_area_m2 / 1e6:.3f} km²")
    print(f"{distance} m land-cover-classified area: {classified_area_m2 / 1e6:.3f} km²")
    print(
        f"{distance} m land-cover coverage of geometric ring: "
        f"{classified_area_m2 / ring_area_m2 * 100:.2f}%"
    )
    print(
        f"{distance} m classified percentage check: "
        f"{summary['% of surrounding ring'].sum():.10f}%"
    )

    display(summary)

    out_csv = output_dir / f"fn_mangroves_surrounding_landuse_{distance}m.csv"
    summary.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")


## Group into encroachment classes

The prompt-level classes are group labels. The source `Classify` field uses more detailed names, so this mapping avoids the exact-string mismatch that would otherwise return 0%.

In [ ]:
encroachment_classes = [
    "Buildings and other infrastructure",
    "Agriculture",
    "Bauxite extraction / quarry",
    "Plantation",
]

encroachment_class_map = {
    "Buildings and other infrastructure": [
        "Buildings and other infrastructures",
    ],
    "Agriculture": [
        "Fields: Herbaceous crops, fallow, cultivated vegetables",
        "Fields: Pasture,Human disturbed, grassland",
        "Fields: Bare Land",
        "Fields and Secondary Forest",
        "Fields or Secondary Forest/Pine Plantation",
        "Fields  and Bamboo",
        "Bamboo and Fields",
    ],
    "Bauxite extraction / quarry": [
        "Bauxite Extraction",
        "Quarry",
    ],
    "Plantation": [
        "Plantation: Tree crops, shrub crops, sugar cane, banana",
        "Hardwood Plantation: Euculytus",
        "Hardwood Plantation: Mahoe",
        "Hardwood Plantation: Mahogany",
        "Hardwood Plantation: Mixed",
    ],
}


def normalise_class_name(value):
    return re.sub(r"\s+", " ", str(value).strip().lower())


normalised_encroachment_lookup = {
    normalise_class_name(source_class): encroachment_class
    for encroachment_class, source_classes in encroachment_class_map.items()
    for source_class in source_classes
}


def assign_encroachment_class(classify_value):
    return normalised_encroachment_lookup.get(normalise_class_name(classify_value))


In [ ]:
encroachment_outputs = {}

for distance, summary in summaries.items():
    summary_with_groups = summary.copy()
    summary_with_groups["encroachment_class"] = summary_with_groups["Classify"].apply(
        assign_encroachment_class
    )

    encroachment_detail = summary_with_groups.dropna(subset=["encroachment_class"]).copy()
    encroachment_detail = encroachment_detail[
        [
            "encroachment_class",
            "Classify",
            "area_m2",
            "area_km2",
            "% of surrounding ring",
            "% of geometric ring",
            "distance_m",
        ]
    ].sort_values(["encroachment_class", "% of surrounding ring"], ascending=[True, False])

    encroachment_by_class = (
        encroachment_detail.groupby("encroachment_class", as_index=False)
        .agg(
            area_m2=("area_m2", "sum"),
            area_km2=("area_km2", "sum"),
            **{
                "% of surrounding ring": ("% of surrounding ring", "sum"),
                "% of geometric ring": ("% of geometric ring", "sum"),
            },
        )
        .sort_values("% of surrounding ring", ascending=False)
    )

    encroachment_by_class["distance_m"] = distance
    encroachment_outputs[distance] = {
        "by_class": encroachment_by_class,
        "detail": encroachment_detail,
    }

    risk_pct = encroachment_by_class["% of surrounding ring"].sum()
    risk_pct_geometric = encroachment_by_class["% of geometric ring"].sum()

    print(
        f"{distance} m ring encroachment-risk share "
        f"of land-cover-classified ring: {risk_pct:.2f}%"
    )
    print(f"{distance} m ring encroachment-risk share of geometric ring: {risk_pct_geometric:.2f}%")
    print("\nEncroachment by requested class:")
    display(encroachment_by_class)
    print("\nUnderlying land-cover classes counted:")
    display(encroachment_detail)

    by_class_csv = output_dir / f"fn_mangroves_encroachment_by_class_{distance}m.csv"
    detail_csv = output_dir / f"fn_mangroves_encroachment_detail_{distance}m.csv"
    encroachment_by_class.to_csv(by_class_csv, index=False)
    encroachment_detail.to_csv(detail_csv, index=False)
    print(f"Saved: {by_class_csv}")
    print(f"Saved: {detail_csv}")


## Encroachment risk for mangrove patches with avoided EAD

This section brings in the weighted area-distance mangrove priority output generated from the `02_mangroves_coastal_flooding` workflow. It calculates a separate 100 m surrounding ring for each mangrove patch, then joins patch-level encroachment exposure to avoided-EAD and net-EAD attribution fields.

Patch rings can overlap, so the cohort totals below are patch-associated surrounding areas rather than a dissolved national ring.

In [ ]:
from shapely import make_valid

weighted_area_distance_priority_path = (
    base_path
    / "dphil_paper_3/results_coastal_scenario_comparison/weighted_area_distance_signed/mangrove_priority_ranking/mangrove_priority_map_weighted_area_distance.gpkg"
)

avoided_ead_mangroves = gpd.read_file(weighted_area_distance_priority_path).to_crs(jamaica_metric_grid_crs)
avoided_ead_mangroves = avoided_ead_mangroves[
    avoided_ead_mangroves.geometry.notna() & ~avoided_ead_mangroves.geometry.is_empty
].copy()

print(weighted_area_distance_priority_path)
print(f"Mangrove patches in avoided-EAD priority layer: {len(avoided_ead_mangroves):,}")

cohort_checks = {
    "avoided_usd_mean > 0": avoided_ead_mangroves["avoided_usd_mean"].fillna(0) > 0,
    "net_usd_mean > 0": avoided_ead_mangroves["net_usd_mean"].fillna(0) > 0,
    "no_regret_net_positive": avoided_ead_mangroves["no_regret_net_positive"].fillna(False).astype(bool),
}

for label, mask in cohort_checks.items():
    cohort = avoided_ead_mangroves.loc[mask]
    print(
        f"{label}: {len(cohort):,} patches | "
        f"{cohort['area_ha'].sum():,.1f} ha | "
        f"mean avoided EAD ${cohort['avoided_usd_mean'].sum():,.0f}/yr"
    )

display(
    avoided_ead_mangroves[
        [
            "Mangrove_ID",
            "Parish",
            "TYPE",
            "area_ha",
            "avoided_usd_min",
            "avoided_usd_max",
            "avoided_usd_mean",
            "net_usd_min",
            "net_usd_max",
            "net_usd_mean",
            "priority_rank",
            "priority_tier",
        ]
    ].head(10)
)


In [ ]:
def summarise_patch_ring_encroachment(mangrove_patches, landcover, distance_m=100):
    spatial_index = landcover.sindex
    patch_rows = []
    detail_parts = []

    for patch_number, patch in enumerate(mangrove_patches.itertuples(index=False), start=1):
        mangrove_id = int(patch.Mangrove_ID)
        patch_geom = patch.geometry
        if patch_geom is None or patch_geom.is_empty:
            continue

        if not patch_geom.is_valid:
            patch_geom = make_valid(patch_geom)

        ring_geom = patch_geom.buffer(distance_m).difference(patch_geom)
        ring_area_m2 = ring_geom.area if ring_geom is not None and not ring_geom.is_empty else 0

        base_row = {
            "Mangrove_ID": mangrove_id,
            "distance_m": distance_m,
            "ring_area_m2": ring_area_m2,
            "classified_ring_area_m2": 0.0,
            "encroachment_area_m2": 0.0,
            "encroachment_pct_of_classified_ring": 0.0,
            "encroachment_pct_of_geometric_ring": 0.0,
            "has_encroachment_risk_100m": False,
        }

        for encroachment_class in encroachment_classes:
            safe_name = normalise_class_name(encroachment_class).replace(" / ", "_").replace(" ", "_").replace("/", "_")
            base_row[f"{safe_name}_area_m2"] = 0.0
            base_row[f"{safe_name}_pct_of_classified_ring"] = 0.0

        if ring_area_m2 == 0:
            patch_rows.append(base_row)
            continue

        candidate_idx = list(spatial_index.query(ring_geom, predicate="intersects"))
        candidates = landcover.iloc[candidate_idx][["Classify", "geometry"]].copy()

        if candidates.empty:
            patch_rows.append(base_row)
            continue

        invalid_candidates = ~candidates.is_valid
        if invalid_candidates.any():
            candidates.loc[invalid_candidates, "geometry"] = candidates.loc[
                invalid_candidates, "geometry"
            ].make_valid()

        intersections = candidates.geometry.intersection(ring_geom)
        keep = intersections.notna() & ~intersections.is_empty

        if not keep.any():
            patch_rows.append(base_row)
            continue

        ring_land = candidates.loc[keep, ["Classify"]].copy()
        ring_land["area_m2"] = intersections.loc[keep].area
        ring_land["encroachment_class"] = ring_land["Classify"].apply(assign_encroachment_class)

        classified_area_m2 = ring_land["area_m2"].sum()
        encroachment_area_m2 = ring_land.loc[
            ring_land["encroachment_class"].notna(), "area_m2"
        ].sum()

        base_row.update(
            {
                "classified_ring_area_m2": classified_area_m2,
                "encroachment_area_m2": encroachment_area_m2,
                "encroachment_pct_of_classified_ring": (
                    encroachment_area_m2 / classified_area_m2 * 100 if classified_area_m2 > 0 else 0
                ),
                "encroachment_pct_of_geometric_ring": (
                    encroachment_area_m2 / ring_area_m2 * 100 if ring_area_m2 > 0 else 0
                ),
                "has_encroachment_risk_100m": encroachment_area_m2 > 0,
            }
        )

        encroachment_by_class = (
            ring_land.dropna(subset=["encroachment_class"])
            .groupby("encroachment_class", as_index=False)["area_m2"]
            .sum()
        )

        for row in encroachment_by_class.itertuples(index=False):
            safe_name = normalise_class_name(row.encroachment_class).replace(" / ", "_").replace(" ", "_").replace("/", "_")
            base_row[f"{safe_name}_area_m2"] = row.area_m2
            base_row[f"{safe_name}_pct_of_classified_ring"] = (
                row.area_m2 / classified_area_m2 * 100 if classified_area_m2 > 0 else 0
            )

        detail = (
            ring_land.dropna(subset=["encroachment_class"])
            .groupby(["encroachment_class", "Classify"], as_index=False)["area_m2"]
            .sum()
        )
        if not detail.empty:
            detail["Mangrove_ID"] = mangrove_id
            detail["distance_m"] = distance_m
            detail["area_km2"] = detail["area_m2"] / 1e6
            detail["pct_of_classified_ring"] = np.where(
                classified_area_m2 > 0,
                detail["area_m2"] / classified_area_m2 * 100,
                0,
            )
            detail["pct_of_geometric_ring"] = np.where(
                ring_area_m2 > 0,
                detail["area_m2"] / ring_area_m2 * 100,
                0,
            )
            detail_parts.append(detail)

        patch_rows.append(base_row)

        if patch_number % 25 == 0 or patch_number == len(mangrove_patches):
            print(f"Processed {patch_number:,}/{len(mangrove_patches):,} mangrove patches")

    patch_summary = pd.DataFrame(patch_rows)
    patch_detail = pd.concat(detail_parts, ignore_index=True) if detail_parts else pd.DataFrame()

    return patch_summary, patch_detail


In [ ]:
patch_encroachment_summary, patch_encroachment_detail = summarise_patch_ring_encroachment(
    avoided_ead_mangroves,
    terrestrial_landcover,
    distance_m=100,
)

ead_columns = [
    "Mangrove_ID",
    "Parish",
    "TYPE",
    "area_ha",
    "avoided_usd_min",
    "avoided_usd_max",
    "avoided_usd_mean",
    "net_usd_min",
    "net_usd_max",
    "net_usd_mean",
    "has_increase_any",
    "has_net_negative_any",
    "no_regret",
    "no_regret_net_positive",
    "priority_rank",
    "priority_tier",
]

patch_encroachment_with_ead = avoided_ead_mangroves[ead_columns].merge(
    patch_encroachment_summary,
    on="Mangrove_ID",
    how="left",
)

patch_encroachment_with_ead["ring_area_km2"] = patch_encroachment_with_ead["ring_area_m2"] / 1e6
patch_encroachment_with_ead["classified_ring_area_km2"] = patch_encroachment_with_ead["classified_ring_area_m2"] / 1e6
patch_encroachment_with_ead["encroachment_area_km2"] = patch_encroachment_with_ead["encroachment_area_m2"] / 1e6

patch_out_csv = output_dir / "fn_mangrove_patch_100m_encroachment_with_avoided_ead.csv"
detail_out_csv = output_dir / "fn_mangrove_patch_100m_encroachment_detail.csv"
patch_encroachment_with_ead.to_csv(patch_out_csv, index=False)
patch_encroachment_detail.to_csv(detail_out_csv, index=False)

print(f"Saved: {patch_out_csv}")
print(f"Saved: {detail_out_csv}")

display(
    patch_encroachment_with_ead.sort_values(
        ["avoided_usd_mean", "encroachment_pct_of_classified_ring"],
        ascending=[False, False],
    ).head(15)
)


In [ ]:
analysis_groups = {
    "avoided_ead_mean_positive": patch_encroachment_with_ead["avoided_usd_mean"].fillna(0) > 0,
    "net_ead_mean_positive": patch_encroachment_with_ead["net_usd_mean"].fillna(0) > 0,
    "no_regret_net_positive": patch_encroachment_with_ead["no_regret_net_positive"].fillna(False).astype(bool),
}

summary_rows = []
class_rows = []

class_area_columns = {
    encroachment_class: normalise_class_name(encroachment_class).replace(" / ", "_").replace(" ", "_").replace("/", "_")
    for encroachment_class in encroachment_classes
}

for group_name, mask in analysis_groups.items():
    cohort = patch_encroachment_with_ead.loc[mask].copy()
    at_risk = cohort[cohort["has_encroachment_risk_100m"].fillna(False)].copy()

    total_mangrove_area_ha = cohort["area_ha"].sum()
    total_avoided_usd_mean = cohort["avoided_usd_mean"].sum()
    total_net_usd_mean = cohort["net_usd_mean"].sum()
    total_classified_ring_area_m2 = cohort["classified_ring_area_m2"].sum()
    total_geometric_ring_area_m2 = cohort["ring_area_m2"].sum()
    total_encroachment_area_m2 = cohort["encroachment_area_m2"].sum()

    summary_rows.append(
        {
            "analysis_group": group_name,
            "patch_count": len(cohort),
            "patches_with_any_100m_encroachment": len(at_risk),
            "pct_patches_with_any_100m_encroachment": len(at_risk) / len(cohort) * 100 if len(cohort) else 0,
            "mangrove_area_ha": total_mangrove_area_ha,
            "mangrove_area_ha_with_any_100m_encroachment": at_risk["area_ha"].sum(),
            "pct_mangrove_area_with_any_100m_encroachment": (
                at_risk["area_ha"].sum() / total_mangrove_area_ha * 100 if total_mangrove_area_ha > 0 else 0
            ),
            "avoided_usd_mean": total_avoided_usd_mean,
            "avoided_usd_mean_with_any_100m_encroachment": at_risk["avoided_usd_mean"].sum(),
            "pct_avoided_usd_mean_with_any_100m_encroachment": (
                at_risk["avoided_usd_mean"].sum() / total_avoided_usd_mean * 100
                if total_avoided_usd_mean > 0 else 0
            ),
            "net_usd_mean": total_net_usd_mean,
            "ring_area_km2_patch_associated": total_geometric_ring_area_m2 / 1e6,
            "classified_ring_area_km2_patch_associated": total_classified_ring_area_m2 / 1e6,
            "encroachment_area_km2_patch_associated": total_encroachment_area_m2 / 1e6,
            "encroachment_pct_of_classified_ring_patch_associated": (
                total_encroachment_area_m2 / total_classified_ring_area_m2 * 100
                if total_classified_ring_area_m2 > 0 else 0
            ),
            "encroachment_pct_of_geometric_ring_patch_associated": (
                total_encroachment_area_m2 / total_geometric_ring_area_m2 * 100
                if total_geometric_ring_area_m2 > 0 else 0
            ),
        }
    )

    for encroachment_class, safe_name in class_area_columns.items():
        class_area_m2 = cohort[f"{safe_name}_area_m2"].sum()
        class_rows.append(
            {
                "analysis_group": group_name,
                "encroachment_class": encroachment_class,
                "area_m2": class_area_m2,
                "area_km2": class_area_m2 / 1e6,
                "pct_of_classified_ring_patch_associated": (
                    class_area_m2 / total_classified_ring_area_m2 * 100
                    if total_classified_ring_area_m2 > 0 else 0
                ),
                "pct_of_geometric_ring_patch_associated": (
                    class_area_m2 / total_geometric_ring_area_m2 * 100
                    if total_geometric_ring_area_m2 > 0 else 0
                ),
            }
        )

avoided_ead_encroachment_summary = pd.DataFrame(summary_rows)
avoided_ead_encroachment_by_class = pd.DataFrame(class_rows).sort_values(
    ["analysis_group", "pct_of_classified_ring_patch_associated"],
    ascending=[True, False],
)

summary_out_csv = output_dir / "fn_mangrove_avoided_ead_encroachment_summary_100m.csv"
class_out_csv = output_dir / "fn_mangrove_avoided_ead_encroachment_by_class_100m.csv"
avoided_ead_encroachment_summary.to_csv(summary_out_csv, index=False)
avoided_ead_encroachment_by_class.to_csv(class_out_csv, index=False)

display(avoided_ead_encroachment_summary)
display(avoided_ead_encroachment_by_class)

print(f"Saved: {summary_out_csv}")
print(f"Saved: {class_out_csv}")


## Minimum and maximum scenario range

The previous section uses the combined priority layer and reports mean min/max values. For scenario-specific reporting, this section reads the explicit minimum and maximum coastal flooding attribution outputs, joins each to the same patch-level 100 m encroachment exposure, and then calculates the range between scenarios.

In [ ]:
scenario_attribution_paths = {
    "minimum": base_path
    / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.gpkg",
    "maximum": base_path
    / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.gpkg",
}


def load_scenario_attribution(scenario_name, path):
    gdf = gpd.read_file(path).to_crs(jamaica_metric_grid_crs)
    keep_columns = [
        "Mangrove_ID",
        "Parish",
        "TYPE",
        "Mangrove_Area_ha",
        "Net_Avoided_EAD_USD_attributed",
        "Positive_Avoided_EAD_USD_attributed",
        "Negative_Avoided_EAD_USD_attributed_abs",
        "Gross_Avoided_EAD_USD_attributed",
        "Asset_Count",
        "Analysis_Unit_Count",
        "Mean_Distance_m",
        "Positive_Asset_Count",
        "Negative_Asset_Count",
        "Net_Impact_Class",
    ]
    out = pd.DataFrame(gdf[keep_columns]).rename(
        columns={
            "Mangrove_Area_ha": "area_ha",
            "Net_Avoided_EAD_USD_attributed": "net_usd",
            "Positive_Avoided_EAD_USD_attributed": "avoided_usd",
            "Negative_Avoided_EAD_USD_attributed_abs": "increase_usd_abs",
            "Gross_Avoided_EAD_USD_attributed": "gross_avoided_usd",
            "Asset_Count": "asset_count",
            "Analysis_Unit_Count": "analysis_unit_count",
            "Mean_Distance_m": "mean_distance_m",
            "Positive_Asset_Count": "positive_asset_count",
            "Negative_Asset_Count": "negative_asset_count",
            "Net_Impact_Class": "net_impact_class",
        }
    )
    out["scenario"] = scenario_name
    return out


scenario_attribution = pd.concat(
    [
        load_scenario_attribution(scenario_name, path)
        for scenario_name, path in scenario_attribution_paths.items()
    ],
    ignore_index=True,
)

scenario_attribution["Mangrove_ID"] = pd.to_numeric(scenario_attribution["Mangrove_ID"]).astype("Int64")
patch_encroachment_for_join = patch_encroachment_summary.copy()
patch_encroachment_for_join["Mangrove_ID"] = pd.to_numeric(
    patch_encroachment_for_join["Mangrove_ID"]
).astype("Int64")

scenario_patch_encroachment = scenario_attribution.merge(
    patch_encroachment_for_join,
    on="Mangrove_ID",
    how="left",
)

scenario_patch_encroachment["ring_area_km2"] = scenario_patch_encroachment["ring_area_m2"] / 1e6
scenario_patch_encroachment["classified_ring_area_km2"] = scenario_patch_encroachment["classified_ring_area_m2"] / 1e6
scenario_patch_encroachment["encroachment_area_km2"] = scenario_patch_encroachment["encroachment_area_m2"] / 1e6
scenario_patch_encroachment["has_positive_avoided_ead"] = scenario_patch_encroachment["avoided_usd"].fillna(0) > 0
scenario_patch_encroachment["has_positive_net_ead"] = scenario_patch_encroachment["net_usd"].fillna(0) > 0

scenario_patch_out_csv = output_dir / "fn_mangrove_patch_100m_encroachment_with_min_max_scenario_ead.csv"
scenario_patch_encroachment.to_csv(scenario_patch_out_csv, index=False)
print(f"Saved: {scenario_patch_out_csv}")

display(
    scenario_patch_encroachment.sort_values(
        ["scenario", "avoided_usd"],
        ascending=[True, False],
    ).head(20)
)


In [ ]:
scenario_analysis_groups = {
    "avoided_ead_positive": lambda df: df["avoided_usd"].fillna(0) > 0,
    "net_ead_positive": lambda df: df["net_usd"].fillna(0) > 0,
}

class_area_columns = {
    encroachment_class: normalise_class_name(encroachment_class)
    .replace(" / ", "_")
    .replace(" ", "_")
    .replace("/", "_")
    for encroachment_class in encroachment_classes
}

scenario_summary_rows = []
scenario_class_rows = []

for scenario_name, scenario_df in scenario_patch_encroachment.groupby("scenario"):
    for group_name, mask_fn in scenario_analysis_groups.items():
        cohort = scenario_df.loc[mask_fn(scenario_df)].copy()
        at_risk = cohort[cohort["has_encroachment_risk_100m"].fillna(False)].copy()

        total_mangrove_area_ha = cohort["area_ha"].sum()
        total_avoided_usd = cohort["avoided_usd"].sum()
        total_net_usd = cohort["net_usd"].sum()
        total_classified_ring_area_m2 = cohort["classified_ring_area_m2"].sum()
        total_geometric_ring_area_m2 = cohort["ring_area_m2"].sum()
        total_encroachment_area_m2 = cohort["encroachment_area_m2"].sum()

        scenario_summary_rows.append(
            {
                "scenario": scenario_name,
                "analysis_group": group_name,
                "patch_count": len(cohort),
                "patches_with_any_100m_encroachment": len(at_risk),
                "pct_patches_with_any_100m_encroachment": len(at_risk) / len(cohort) * 100 if len(cohort) else 0,
                "mangrove_area_ha": total_mangrove_area_ha,
                "mangrove_area_ha_with_any_100m_encroachment": at_risk["area_ha"].sum(),
                "pct_mangrove_area_with_any_100m_encroachment": (
                    at_risk["area_ha"].sum() / total_mangrove_area_ha * 100 if total_mangrove_area_ha > 0 else 0
                ),
                "avoided_usd": total_avoided_usd,
                "avoided_usd_with_any_100m_encroachment": at_risk["avoided_usd"].sum(),
                "pct_avoided_usd_with_any_100m_encroachment": (
                    at_risk["avoided_usd"].sum() / total_avoided_usd * 100 if total_avoided_usd > 0 else 0
                ),
                "net_usd": total_net_usd,
                "net_usd_with_any_100m_encroachment": at_risk["net_usd"].sum(),
                "pct_net_usd_with_any_100m_encroachment": (
                    at_risk["net_usd"].sum() / total_net_usd * 100 if total_net_usd > 0 else 0
                ),
                "ring_area_km2_patch_associated": total_geometric_ring_area_m2 / 1e6,
                "classified_ring_area_km2_patch_associated": total_classified_ring_area_m2 / 1e6,
                "encroachment_area_km2_patch_associated": total_encroachment_area_m2 / 1e6,
                "encroachment_pct_of_classified_ring_patch_associated": (
                    total_encroachment_area_m2 / total_classified_ring_area_m2 * 100
                    if total_classified_ring_area_m2 > 0 else 0
                ),
                "encroachment_pct_of_geometric_ring_patch_associated": (
                    total_encroachment_area_m2 / total_geometric_ring_area_m2 * 100
                    if total_geometric_ring_area_m2 > 0 else 0
                ),
            }
        )

        for encroachment_class, safe_name in class_area_columns.items():
            class_area_m2 = cohort[f"{safe_name}_area_m2"].sum()
            scenario_class_rows.append(
                {
                    "scenario": scenario_name,
                    "analysis_group": group_name,
                    "encroachment_class": encroachment_class,
                    "area_m2": class_area_m2,
                    "area_km2": class_area_m2 / 1e6,
                    "pct_of_classified_ring_patch_associated": (
                        class_area_m2 / total_classified_ring_area_m2 * 100
                        if total_classified_ring_area_m2 > 0 else 0
                    ),
                    "pct_of_geometric_ring_patch_associated": (
                        class_area_m2 / total_geometric_ring_area_m2 * 100
                        if total_geometric_ring_area_m2 > 0 else 0
                    ),
                }
            )

scenario_encroachment_summary = pd.DataFrame(scenario_summary_rows).sort_values(
    ["analysis_group", "scenario"]
)
scenario_encroachment_by_class = pd.DataFrame(scenario_class_rows).sort_values(
    ["analysis_group", "scenario", "pct_of_classified_ring_patch_associated"],
    ascending=[True, True, False],
)


def make_scenario_range_table(df, index_columns):
    minimum = df[df["scenario"] == "minimum"].set_index(index_columns)
    maximum = df[df["scenario"] == "maximum"].set_index(index_columns)
    common_index = minimum.index.intersection(maximum.index)
    numeric_columns = [
        col
        for col in df.select_dtypes(include="number").columns
        if col not in index_columns
    ]

    rows = []
    for index_value in common_index:
        index_values = index_value if isinstance(index_value, tuple) else (index_value,)
        base = dict(zip(index_columns, index_values))
        for metric in numeric_columns:
            minimum_value = minimum.loc[index_value, metric]
            maximum_value = maximum.loc[index_value, metric]
            rows.append(
                {
                    **base,
                    "metric": metric,
                    "minimum_scenario_value": minimum_value,
                    "maximum_scenario_value": maximum_value,
                    "scenario_range": maximum_value - minimum_value,
                }
            )
    return pd.DataFrame(rows)


scenario_encroachment_summary_range = make_scenario_range_table(
    scenario_encroachment_summary,
    ["analysis_group"],
)
scenario_encroachment_by_class_range = make_scenario_range_table(
    scenario_encroachment_by_class,
    ["analysis_group", "encroachment_class"],
)

scenario_summary_out_csv = output_dir / "fn_mangrove_min_max_scenario_encroachment_summary_100m.csv"
scenario_class_out_csv = output_dir / "fn_mangrove_min_max_scenario_encroachment_by_class_100m.csv"
scenario_summary_range_out_csv = output_dir / "fn_mangrove_min_max_scenario_encroachment_summary_range_100m.csv"
scenario_class_range_out_csv = output_dir / "fn_mangrove_min_max_scenario_encroachment_by_class_range_100m.csv"

scenario_encroachment_summary.to_csv(scenario_summary_out_csv, index=False)
scenario_encroachment_by_class.to_csv(scenario_class_out_csv, index=False)
scenario_encroachment_summary_range.to_csv(scenario_summary_range_out_csv, index=False)
scenario_encroachment_by_class_range.to_csv(scenario_class_range_out_csv, index=False)

display(scenario_encroachment_summary)
display(scenario_encroachment_by_class)
display(
    scenario_encroachment_summary_range[
        scenario_encroachment_summary_range["metric"].isin(
            [
                "patch_count",
                "patches_with_any_100m_encroachment",
                "mangrove_area_ha_with_any_100m_encroachment",
                "pct_avoided_usd_with_any_100m_encroachment",
                "encroachment_pct_of_classified_ring_patch_associated",
            ]
        )
    ]
)

print(f"Saved: {scenario_summary_out_csv}")
print(f"Saved: {scenario_class_out_csv}")
print(f"Saved: {scenario_summary_range_out_csv}")
print(f"Saved: {scenario_class_range_out_csv}")


## Direct edge pressure: 10 m surrounding ring

The 100 m analysis captures the surrounding landscape context. This section uses a 10 m outside ring as a direct edge-pressure sensitivity: close enough to represent land uses bordering the mangrove margin, but less brittle than exact boundary-touch tests or a 5 m ring.

In [ ]:
edge_distance_m = 10

edge_summary, edge_ring_area_m2, edge_classified_area_m2 = summarise_surrounding_landuse(
    mangroves_fon,
    terrestrial_landcover,
    edge_distance_m,
)
edge_summary["encroachment_class"] = edge_summary["Classify"].apply(assign_encroachment_class)

edge_encroachment_detail = edge_summary.dropna(subset=["encroachment_class"]).copy()
edge_encroachment_by_class = (
    edge_encroachment_detail.groupby("encroachment_class", as_index=False)
    .agg(
        area_m2=("area_m2", "sum"),
        area_km2=("area_km2", "sum"),
        **{
            "% of surrounding ring": ("% of surrounding ring", "sum"),
            "% of geometric ring": ("% of geometric ring", "sum"),
        },
    )
    .sort_values("% of surrounding ring", ascending=False)
)
edge_encroachment_by_class["distance_m"] = edge_distance_m

edge_risk_pct = edge_encroachment_by_class["% of surrounding ring"].sum()
edge_risk_pct_geometric = edge_encroachment_by_class["% of geometric ring"].sum()

print(f"{edge_distance_m} m edge ring area: {edge_ring_area_m2 / 1e6:.3f} km²")
print(f"{edge_distance_m} m land-cover-classified edge area: {edge_classified_area_m2 / 1e6:.3f} km²")
print(
    f"{edge_distance_m} m edge encroachment-risk share of land-cover-classified edge ring: "
    f"{edge_risk_pct:.2f}%"
)
print(
    f"{edge_distance_m} m edge encroachment-risk share of geometric edge ring: "
    f"{edge_risk_pct_geometric:.2f}%"
)

display(edge_summary)
display(edge_encroachment_by_class)

edge_landuse_out_csv = output_dir / f"fn_mangroves_edge_surrounding_landuse_{edge_distance_m}m.csv"
edge_by_class_out_csv = output_dir / f"fn_mangroves_edge_encroachment_by_class_{edge_distance_m}m.csv"
edge_summary.to_csv(edge_landuse_out_csv, index=False)
edge_encroachment_by_class.to_csv(edge_by_class_out_csv, index=False)
print(f"Saved: {edge_landuse_out_csv}")
print(f"Saved: {edge_by_class_out_csv}")


In [ ]:
edge_patch_encroachment_summary, edge_patch_encroachment_detail = summarise_patch_ring_encroachment(
    avoided_ead_mangroves,
    terrestrial_landcover,
    distance_m=edge_distance_m,
)

edge_flag_col = f"has_encroachment_risk_{edge_distance_m}m"
edge_patch_encroachment_summary = edge_patch_encroachment_summary.rename(
    columns={"has_encroachment_risk_100m": edge_flag_col}
)
edge_patch_encroachment_summary["edge_ring_area_km2"] = edge_patch_encroachment_summary["ring_area_m2"] / 1e6
edge_patch_encroachment_summary["classified_edge_ring_area_km2"] = edge_patch_encroachment_summary["classified_ring_area_m2"] / 1e6
edge_patch_encroachment_summary["edge_encroachment_area_km2"] = edge_patch_encroachment_summary["encroachment_area_m2"] / 1e6

edge_patch_detail_out_csv = output_dir / f"fn_mangrove_patch_edge_{edge_distance_m}m_encroachment_detail.csv"
edge_patch_summary_out_csv = output_dir / f"fn_mangrove_patch_edge_{edge_distance_m}m_encroachment_summary.csv"
edge_patch_encroachment_detail.to_csv(edge_patch_detail_out_csv, index=False)
edge_patch_encroachment_summary.to_csv(edge_patch_summary_out_csv, index=False)

print(f"Saved: {edge_patch_summary_out_csv}")
print(f"Saved: {edge_patch_detail_out_csv}")
display(edge_patch_encroachment_summary.sort_values("encroachment_pct_of_classified_ring", ascending=False).head(15))


In [ ]:
edge_patch_encroachment_for_join = edge_patch_encroachment_summary.copy()
edge_patch_encroachment_for_join["Mangrove_ID"] = pd.to_numeric(
    edge_patch_encroachment_for_join["Mangrove_ID"]
).astype("Int64")

edge_scenario_patch_encroachment = scenario_attribution.merge(
    edge_patch_encroachment_for_join,
    on="Mangrove_ID",
    how="left",
)

edge_scenario_patch_encroachment["has_positive_avoided_ead"] = edge_scenario_patch_encroachment[
    "avoided_usd"
].fillna(0) > 0
edge_scenario_patch_encroachment["has_positive_net_ead"] = edge_scenario_patch_encroachment[
    "net_usd"
].fillna(0) > 0

edge_scenario_patch_out_csv = output_dir / f"fn_mangrove_patch_edge_{edge_distance_m}m_encroachment_with_min_max_scenario_ead.csv"
edge_scenario_patch_encroachment.to_csv(edge_scenario_patch_out_csv, index=False)
print(f"Saved: {edge_scenario_patch_out_csv}")

display(
    edge_scenario_patch_encroachment.sort_values(
        ["scenario", "avoided_usd"],
        ascending=[True, False],
    ).head(20)
)


In [ ]:
edge_summary_rows = []
edge_class_rows = []

for scenario_name, scenario_df in edge_scenario_patch_encroachment.groupby("scenario"):
    for group_name, mask_fn in scenario_analysis_groups.items():
        cohort = scenario_df.loc[mask_fn(scenario_df)].copy()
        at_risk = cohort[cohort[edge_flag_col].fillna(False)].copy()

        total_mangrove_area_ha = cohort["area_ha"].sum()
        total_avoided_usd = cohort["avoided_usd"].sum()
        total_net_usd = cohort["net_usd"].sum()
        total_classified_ring_area_m2 = cohort["classified_ring_area_m2"].sum()
        total_geometric_ring_area_m2 = cohort["ring_area_m2"].sum()
        total_encroachment_area_m2 = cohort["encroachment_area_m2"].sum()

        edge_summary_rows.append(
            {
                "scenario": scenario_name,
                "analysis_group": group_name,
                "edge_distance_m": edge_distance_m,
                "patch_count": len(cohort),
                "patches_with_any_edge_encroachment": len(at_risk),
                "pct_patches_with_any_edge_encroachment": len(at_risk) / len(cohort) * 100 if len(cohort) else 0,
                "mangrove_area_ha": total_mangrove_area_ha,
                "mangrove_area_ha_with_any_edge_encroachment": at_risk["area_ha"].sum(),
                "pct_mangrove_area_with_any_edge_encroachment": (
                    at_risk["area_ha"].sum() / total_mangrove_area_ha * 100 if total_mangrove_area_ha > 0 else 0
                ),
                "avoided_usd": total_avoided_usd,
                "avoided_usd_with_any_edge_encroachment": at_risk["avoided_usd"].sum(),
                "pct_avoided_usd_with_any_edge_encroachment": (
                    at_risk["avoided_usd"].sum() / total_avoided_usd * 100 if total_avoided_usd > 0 else 0
                ),
                "net_usd": total_net_usd,
                "net_usd_with_any_edge_encroachment": at_risk["net_usd"].sum(),
                "pct_net_usd_with_any_edge_encroachment": (
                    at_risk["net_usd"].sum() / total_net_usd * 100 if total_net_usd > 0 else 0
                ),
                "edge_ring_area_km2_patch_associated": total_geometric_ring_area_m2 / 1e6,
                "classified_edge_ring_area_km2_patch_associated": total_classified_ring_area_m2 / 1e6,
                "edge_encroachment_area_km2_patch_associated": total_encroachment_area_m2 / 1e6,
                "edge_encroachment_pct_of_classified_ring_patch_associated": (
                    total_encroachment_area_m2 / total_classified_ring_area_m2 * 100
                    if total_classified_ring_area_m2 > 0 else 0
                ),
                "edge_encroachment_pct_of_geometric_ring_patch_associated": (
                    total_encroachment_area_m2 / total_geometric_ring_area_m2 * 100
                    if total_geometric_ring_area_m2 > 0 else 0
                ),
            }
        )

        for encroachment_class, safe_name in class_area_columns.items():
            class_area_m2 = cohort[f"{safe_name}_area_m2"].sum()
            edge_class_rows.append(
                {
                    "scenario": scenario_name,
                    "analysis_group": group_name,
                    "edge_distance_m": edge_distance_m,
                    "encroachment_class": encroachment_class,
                    "area_m2": class_area_m2,
                    "area_km2": class_area_m2 / 1e6,
                    "pct_of_classified_edge_ring_patch_associated": (
                        class_area_m2 / total_classified_ring_area_m2 * 100
                        if total_classified_ring_area_m2 > 0 else 0
                    ),
                    "pct_of_geometric_edge_ring_patch_associated": (
                        class_area_m2 / total_geometric_ring_area_m2 * 100
                        if total_geometric_ring_area_m2 > 0 else 0
                    ),
                }
            )

edge_scenario_encroachment_summary = pd.DataFrame(edge_summary_rows).sort_values(
    ["analysis_group", "scenario"]
)
edge_scenario_encroachment_by_class = pd.DataFrame(edge_class_rows).sort_values(
    ["analysis_group", "scenario", "pct_of_classified_edge_ring_patch_associated"],
    ascending=[True, True, False],
)

edge_scenario_encroachment_summary_range = make_scenario_range_table(
    edge_scenario_encroachment_summary,
    ["analysis_group"],
)
edge_scenario_encroachment_by_class_range = make_scenario_range_table(
    edge_scenario_encroachment_by_class,
    ["analysis_group", "encroachment_class"],
)

edge_summary_out_csv = output_dir / f"fn_mangrove_edge_{edge_distance_m}m_min_max_scenario_encroachment_summary.csv"
edge_class_out_csv = output_dir / f"fn_mangrove_edge_{edge_distance_m}m_min_max_scenario_encroachment_by_class.csv"
edge_summary_range_out_csv = output_dir / f"fn_mangrove_edge_{edge_distance_m}m_min_max_scenario_encroachment_summary_range.csv"
edge_class_range_out_csv = output_dir / f"fn_mangrove_edge_{edge_distance_m}m_min_max_scenario_encroachment_by_class_range.csv"

edge_scenario_encroachment_summary.to_csv(edge_summary_out_csv, index=False)
edge_scenario_encroachment_by_class.to_csv(edge_class_out_csv, index=False)
edge_scenario_encroachment_summary_range.to_csv(edge_summary_range_out_csv, index=False)
edge_scenario_encroachment_by_class_range.to_csv(edge_class_range_out_csv, index=False)

display(edge_scenario_encroachment_summary)
display(edge_scenario_encroachment_by_class)
display(
    edge_scenario_encroachment_summary_range[
        edge_scenario_encroachment_summary_range["metric"].isin(
            [
                "patch_count",
                "patches_with_any_edge_encroachment",
                "mangrove_area_ha_with_any_edge_encroachment",
                "pct_avoided_usd_with_any_edge_encroachment",
                "edge_encroachment_pct_of_classified_ring_patch_associated",
            ]
        )
    ]
)

print(f"Saved: {edge_summary_out_csv}")
print(f"Saved: {edge_class_out_csv}")
print(f"Saved: {edge_summary_range_out_csv}")
print(f"Saved: {edge_class_range_out_csv}")
